# PyRadtran Sea Ice Validation Notebook

This notebook demonstrates the validation of the PyRadtran package for sea ice simulations,
focusing on recreating the functionality from the original disort.py script in a more modular way.

**Date:** 6 May 2025

## 1. Setup and Initial Configuration

First, let's ensure the package is properly installed and import the necessary libraries.

In [ ]:
# Import standard libraries first
import os
import sys
import tempfile
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import logging
import traceback
from pathlib import Path
from datetime import datetime, timedelta

# Configure basic logging
logging.basicConfig(level=logging.INFO, 
                   format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# Add parent directory to path if needed
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    print(f"Added {module_path} to sys.path")

# Check package structure
!find {module_path} -type d -not -path "*/\.*" | sort

# Ensure matplotlib figures are displayed at a good size
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Now try to import PyRadtran modules
try:
    import pyradtran
    from pyradtran.config import PathsConfig, SimulationDefaults, SimulationConfig, ExecutionConfig, OutputConfig, load_config
    from pyradtran.core import Simulation
    from pyradtran.io import parse_uvspec_output
    from pyradtran.exceptions import PyRadtranError
    print("Successfully imported PyRadtran modules")
except ImportError as e:
    print(f"Error importing PyRadtran: {e}")
    print("\nTrying to fix package structure...")
    
    # If there's an import error, we may need to ensure the package structure is correct
    # This is a workaround based on the pip install error
    !mkdir -p {module_path}/pyradtran
    !touch {module_path}/pyradtran/__init__.py
    
    # Move key modules into the pyradtran/ directory if they're at the root
    for module in ['config.py', 'core.py', 'exceptions.py', 'helpers.py', 'interface.py', 'io.py', 'utils.py']:
        if os.path.exists(f"{module_path}/{module}") and not os.path.exists(f"{module_path}/pyradtran/{module}"):
            !cp {module_path}/{module} {module_path}/pyradtran/
            print(f"Copied {module} to pyradtran/ directory")
    
    # Try importing again
    try:
        import pyradtran
        from pyradtran.config import PathsConfig, SimulationDefaults, SimulationConfig, ExecutionConfig, OutputConfig, load_config
        from pyradtran.core import Simulation
        from pyradtran.io import parse_uvspec_output
        from pyradtran.exceptions import PyRadtranError
        print("Successfully imported PyRadtran modules after fixes")
    except ImportError as e:
        print(f"Still having import issues: {e}")
        print("Please check the package structure manually.")

### 1.1 Define LibRadtran Paths

Set up the paths to the LibRadtran installation and related files. These paths match your environment.

In [ ]:
# Define LibRadtran and data paths (same as in disort.py)
LIBRADTRAN_DATA_PATH = '/opt/libradtran/2.0.4/share/libRadtran/data'
LIBRADTRAN_EXEC_PATH = '/opt/libradtran/2.0.4/bin/uvspec'
ATMOSPHERE_FILE = '/projekt_agmwend/data/HALO-AC3/05_VELOX_Tools/add_data/afglsw.dat'
SOLAR_SPECTRUM_FILE = '/projekt_agmwend/home_rad/sophie/libradtran/solar_flux/NewGuey2003.dat'
RADIOSONDE_BASE_PATH = '/projekt_agmwend/data/HALO-AC3/01_soundings/RS_for_libradtran/Dropsondes_HALO/'
SIMULATION_OUTPUT_DIR = '/projekt_agmwend/home_rad/Joshua/HALO-AC3_Arctic_leads/data/simulation/disort/'
WORKING_DIR = os.path.join(module_path, 'work')

# Create working directory if it doesn't exist
os.makedirs(WORKING_DIR, exist_ok=True)

# Sea Ice simulation constants
FIXED_OZONE_DU = 300.0  # Fixed total column ozone in Dobson Units
FIXED_IWV_MM = 2.0      # Fixed total column water vapor (precipitable water) in mm
FIXED_SURFACE_TEMP_K = 250.0  # Fixed surface temperature for sea ice conditions (Kelvin)
SEA_ICE_BRDF_TYPE = 20  # RPV BRDF type for sea ice

# Check if LibRadtran is available
libradtran_available = os.path.isfile(LIBRADTRAN_EXEC_PATH) and os.path.isdir(LIBRADTRAN_DATA_PATH)
print(f"LibRadtran available: {libradtran_available}")

if not libradtran_available:
    paths_exist = {
        "LIBRADTRAN_EXEC": os.path.isfile(LIBRADTRAN_EXEC_PATH),
        "LIBRADTRAN_DATA": os.path.isdir(LIBRADTRAN_DATA_PATH),
        "ATMOSPHERE_FILE": os.path.isfile(ATMOSPHERE_FILE),
        "SOLAR_SPECTRUM_FILE": os.path.isfile(SOLAR_SPECTRUM_FILE),
        "RADIOSONDE_BASE_PATH": os.path.isdir(RADIOSONDE_BASE_PATH)
    }
    print("\nPath check results:")
    for path_name, exists in paths_exist.items():
        print(f"{path_name}: {'Found' if exists else 'Not found'}")
    print("\nWARNING: LibRadtran is not available at the specified paths. Some cells may not execute properly.")

### 1.2 Create Sea Ice Configuration

Create a specialized configuration for sea ice simulations based on the settings from disort.py.

In [ ]:
def create_sea_ice_config():
    """Create a configuration for sea ice simulations similar to disort.py"""
    return SimulationConfig(
        paths=PathsConfig(
            libradtran_bin=Path(LIBRADTRAN_EXEC_PATH),
            libradtran_data=Path(LIBRADTRAN_DATA_PATH),
            atmosphere_profile=Path(ATMOSPHERE_FILE),
            solar_spectrum=Path(SOLAR_SPECTRUM_FILE),
            radiosonde_base=Path(RADIOSONDE_BASE_PATH),
            output_dir=Path(SIMULATION_OUTPUT_DIR),
            working_dir=Path(WORKING_DIR)  # Use dedicated working directory
        ),
        simulation_defaults=SimulationDefaults(
            rte_solver='disort',  # Use disort for accuracy in sea ice simulations
            mol_abs_param='lowtran per_nm',  # Same as disort.py
            wavelength_nm=[400, 3600],  # Same range as disort.py
            output_columns=['sza', 'edir', 'eglo', 'edn', 'eup', 'enet', 'esum', 'albedo'],
            output_altitudes_km=[0.0],  # Surface level only
            
            # Surface properties
            albedo_type='library',
            albedo_library='IGBP',
            brdf_type='rpv',
            brdf_rpv_type=SEA_ICE_BRDF_TYPE,  # Sea ice BRDF
            surface_temperature_k=FIXED_SURFACE_TEMP_K,
            
            # Fixed atmospheric composition
            mol_modify={
                'O3': {'value': FIXED_OZONE_DU, 'unit': 'DU'},
                'H2O': {'value': FIXED_IWV_MM, 'unit': 'MM'}
            },
            
            # Default aerosols
            aerosols={
                'enabled': True,
                'aerosol_type': 'default'
            },
            
            # No clouds by default
            clouds={
                'enabled': False
            }
        ),
        # Add the missing required parameters
        execution=ExecutionConfig(
            max_workers=1,  # Single worker for debugging
            cleanup_temp_files=True,  # Clean up temporary files after simulation
            debug_mode=True,  # Enable debug mode to see what's happening
            timeout_seconds=300  # 5-minute timeout for simulations
        ),
        output=OutputConfig(
            filename_prefix="sea_ice_sim",  # Prefix for output files
            filename_suffix="_results.nc",  # Suffix for output files
            netcdf_encoding={"zlib": True, "complevel": 5}  # Enable compression for NetCDF files
        )
    )

# Create and display the sea ice configuration
try:
    sea_ice_config = create_sea_ice_config()
    print("Sea ice configuration created successfully!")
    
    print(f"\nPaths:")
    for key, value in vars(sea_ice_config.paths).items():
        print(f"  {key}: {value}")
    
    print(f"\nSimulation Defaults (selected):")
    print(f"  rte_solver: {sea_ice_config.simulation_defaults.rte_solver}")
    print(f"  mol_abs_param: {sea_ice_config.simulation_defaults.mol_abs_param}")
    print(f"  wavelength_nm: {sea_ice_config.simulation_defaults.wavelength_nm}")
    print(f"  output_columns: {sea_ice_config.simulation_defaults.output_columns}")
    print(f"  surface_temperature_k: {sea_ice_config.simulation_defaults.surface_temperature_k}")
except Exception as e:
    print(f"Error creating configuration: {e}")

## 2. Manual Simulation Test with Core API

First, let's test the core API directly to ensure it works properly.

In [ ]:
# Skip this cell if LibRadtran is not available
if not libradtran_available:
    print("Skipping: LibRadtran not available")
else:
    # Create a test datetime and coordinates
    test_time = pd.Timestamp('2022-04-01 10:00:00').to_pydatetime()  
    test_lat = 75.0  # Arctic location
    test_lon = 0.0   # Prime meridian
    
    print(f"Running manual simulation for {test_time} at lat={test_lat}, lon={test_lon}")
    
    try:
        # Set up detailed logging
        logging.getLogger('pyradtran').setLevel(logging.DEBUG)
        
        # Initialize the simulation runner with our config
        runner = Simulation(sea_ice_config)
        
        # Run the simulation directly
        output_file = runner.run(test_time, test_lat, test_lon)
        
        if output_file and output_file.exists():
            print(f"Simulation succeeded! Output file: {output_file}")
            
            # Parse the output
            result = parse_uvspec_output(output_file, sea_ice_config)
            
            # Display results
            print("\nSimulation Results:")
            for key, value in result.items():
                if not key.startswith('_') and not isinstance(value, dict):
                    if isinstance(value, list) and len(value) > 0:
                        print(f"{key}: {value[0]}")
                    else:
                        print(f"{key}: {value}")
        else:
            print("Simulation failed to produce output file")
            
    except Exception as e:
        print(f"Error in manual simulation: {e}")
        print(traceback.format_exc())